In [ ]:
from lsst.rsp import get_tap_service
import sqlite3
import pandas as pd

In [ ]:
service = get_tap_service("tap")

In [ ]:
test_db_fname = "testing_database-dp2.db"
cnx = sqlite3.connect(test_db_fname)

In [ ]:
ssoid = 21164741220119127

In [ ]:
filter_name = "r"

In [ ]:
diasource_sql_query = f"""
            SELECT
                *
            FROM
                dp2.DiaSource
            WHERE
                ssObjectId = {ssoid}
            """

In [ ]:
diatable = service.search(diasource_sql_query).to_table().to_pandas()

In [ ]:
diatable

In [ ]:
sssource_sql_query = f"""
            SELECT
                *
            FROM
                dp2.SSSource
            WHERE
                ssObjectId = {ssoid}
            """

In [ ]:
sssource_table = service.search(sssource_sql_query).to_table().to_pandas()

In [ ]:
sssource_table.columns

In [ ]:
ssobject_sql_query = f"""
            SELECT
                *
            FROM
                dp2.SSObject
            WHERE
                ssObjectId = {ssoid}
            """

In [ ]:
ssobject_table = service.search(ssobject_sql_query).to_table().to_pandas()

In [ ]:
ssobject_table.columns

In [ ]:
# mpcorb_sql_query = f"""
#             SELECT
#                 *
#             FROM
#                 dp2.mpc_orbits
#             WHERE
#                 ssObjectId = {ssoid}
#             """

mpcorb_sql_query = """SELECT mpc.*
        FROM dp2.mpc_orbits as mpc 
        INNER JOIN dp2.SSObject as sso
        ON mpc.designation = sso.designation
        WHERE sso.ssObjectId = {}""".format(ssoid)

In [ ]:
mpcorb_table = service.search(mpcorb_sql_query).to_table().to_pandas()

In [ ]:
mpcorb_table

In [ ]:
diatable.to_sql("diaSource", con=cnx, if_exists="append", index=False)

In [ ]:
sssource_table.to_sql("ssSource", con=cnx, if_exists="append", index=False)

In [ ]:
ssobject_table.to_sql("ssObject", con=cnx, if_exists="append", index=False)

In [ ]:
mpcorb_table.to_sql("mpc_orbits", con=cnx, if_exists="append", index=False)

In [ ]:
cnx.close()

Testing everything went correctly...

In [ ]:
cnx = sqlite3.connect(test_db_fname)

In [ ]:
example_query = f"""
                SELECT
                    ssObject.ssObjectId, psfFlux, psfFluxErr, band, midpointMjdTai, ra, dec, phaseAngle,
                    topoRange, helioRange, helio_x
                FROM
                    ssObject
                    JOIN diaSource ON ssObject.ssObjectId   = diaSource.ssObjectId
                    JOIN ssSource  ON diaSource.diaSourceId = ssSource.diaSourceId
                WHERE
                    ssObject.ssObjectId = {ssoid} and band = '{filter_name}'
                """

In [ ]:
pd.read_sql_query(example_query, cnx)

In [ ]:
cur = cnx.cursor()

In [ ]:
res = cur.execute("SELECT * FROM sqlite_schema")

In [ ]:
res.fetchall()